In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print("Loading V3 Dataset...")
# Load the processed V3 features
df = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

feature_cols = [
    'year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 
    'is_holiday_season', 'lag_1', 'lag_7', 'rolling_mean_7', 
    'rolling_mean_30', 'is_promo', 'is_stockout'
]

# Explicitly flag categorical columns for CatBoost
cat_features_indices = [
    'year', 'month', 'day_of_week', 'is_weekend', 
    'is_back_to_school', 'is_holiday_season', 'is_promo', 'is_stockout'
]

# Ensure categorical columns are integers
for col in cat_features_indices:
    df[col] = df[col].astype(int)

all_results_ensemble = []

for sku in df["sku_id"].unique():
    print(f"\n--- Training Ensemble for {sku} ---")
    sku_df = df[df["sku_id"] == sku].sort_values('date').reset_index(drop=True)
    
    # 1. CHRONOLOGICAL SPLIT (60% Base Train, 20% Meta Train, 20% Final Test)
    # We need a holdout set to train the Meta-Model so it doesn't overfit
    train_idx = int(len(sku_df) * 0.6)
    meta_idx = int(len(sku_df) * 0.8)
    
    train_df = sku_df.iloc[:train_idx]
    meta_df = sku_df.iloc[train_idx:meta_idx]
    test_df = sku_df.iloc[meta_idx:]
    
    X_train, y_train = train_df[feature_cols], train_df["units_sold_diff"]
    X_meta, y_meta = meta_df[feature_cols], meta_df["units_sold_diff"]
    X_test, y_test_actual, test_lag_1 = test_df[feature_cols], test_df["units_sold"].values, test_df["lag_1"].values
    
    # 2. INITIALIZE LEVEL 0 BASE MODELS
    model_cat = CatBoostRegressor(iterations=150, learning_rate=0.1, depth=6, verbose=0)
    model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, verbosity=0)
    model_lgb = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
    
    # Train Base Models on the 60% training set
    print("Training Level 0 Models (CatBoost, XGBoost, LightGBM)...")
    model_cat.fit(X_train, y_train, cat_features=cat_features_indices)
    model_xgb.fit(X_train, y_train)
    model_lgb.fit(X_train, y_train)
    
    # 3. GENERATE PREDICTIONS FOR THE META-MODEL
    # Base models predict on the 20% Meta Set
    meta_preds_cat = model_cat.predict(X_meta)
    meta_preds_xgb = model_xgb.predict(X_meta)
    meta_preds_lgb = model_lgb.predict(X_meta)
    
    # Create the training dataset for the Level 1 Meta-Model
    X_meta_train = np.column_stack((meta_preds_cat, meta_preds_xgb, meta_preds_lgb))
    
    # 4. TRAIN LEVEL 1 META-MODEL (Linear Regression)
    print("Training Level 1 Meta-Model (Linear Regression)...")
    meta_model = LinearRegression()
    meta_model.fit(X_meta_train, y_meta)
    
    # Print the weights it assigned to each model
    print(f"Algorithm Weights -> CatBoost: {meta_model.coef_[0]:.2f}, XGBoost: {meta_model.coef_[1]:.2f}, LightGBM: {meta_model.coef_[2]:.2f}")
    
    # 5. FINAL EVALUATION ON THE 20% TEST SET
    test_preds_cat = model_cat.predict(X_test)
    test_preds_xgb = model_xgb.predict(X_test)
    test_preds_lgb = model_lgb.predict(X_test)
    
    X_test_meta = np.column_stack((test_preds_cat, test_preds_xgb, test_preds_lgb))
    
    # The Meta-Model makes the final unified prediction on the differences
    final_preds_diff = meta_model.predict(X_test_meta)
    
    # Reconstruct actual sales and apply a floor of 0
    final_preds_actual = np.maximum(0, test_lag_1 + final_preds_diff)
    
    # Calculate Final Metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual, final_preds_actual))
    mae = mean_absolute_error(y_test_actual, final_preds_actual)
    mape = np.mean(np.abs((y_test_actual - final_preds_actual) / y_test_actual)) * 100
    r2 = r2_score(y_test_actual, final_preds_actual)
    
    all_results_ensemble.append({
        "model": "Stacking_Ensemble_V3",
        "sku_id": sku,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

# 6. PRINT FINAL REPORT
print("\n=== FINAL ENSEMBLE PERFORMANCE ===")
summary_df = pd.DataFrame(all_results_ensemble).groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
print(summary_df)

Loading V3 Dataset...

--- Training Ensemble for MBA13 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.29, XGBoost: 0.83, LightGBM: -0.01

--- Training Ensemble for MBA15 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.89, XGBoost: 0.52, LightGBM: -0.37

--- Training Ensemble for MBP14 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.87, XGBoost: 0.78, LightGBM: -0.49

--- Training Ensemble for MBP16 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 1.06, XGBoost: 0.06, LightGBM: -0.13

=== FINAL ENSEMBLE PERFORMANCE ===
                        RMSE     MAE  MAPE     R2
model                                   

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

print("Loading 1-Million-Row Dataset...")
df = pd.read_csv("../data/processed/demand_features_v4_1M.csv", parse_dates=["date"])

# 1. ENCODING FOR COMPATIBILITY
# XGBoost and LightGBM need numerical encoding for stores
le = LabelEncoder()
df['store_id_encoded'] = le.fit_transform(df['store_id'])

# CatBoost prefers string categoricals
df['store_id_str'] = df['store_id'].astype(str)

feature_cols_xgb_lgb = [
    'store_id_encoded', 'year', 'month', 'day_of_week', 'is_weekend', 
    'is_holiday_season', 'lag_1', 'lag_7', 'rolling_mean_7', 
    'rolling_mean_30', 'is_promo', 'is_stockout'
]

feature_cols_cat = [
    'store_id_str', 'year', 'month', 'day_of_week', 'is_weekend', 
    'is_holiday_season', 'lag_1', 'lag_7', 'rolling_mean_7', 
    'rolling_mean_30', 'is_promo', 'is_stockout'
]

cat_features_indices = ['store_id_str', 'year', 'month', 'day_of_week', 'is_weekend', 'is_holiday_season', 'is_promo', 'is_stockout']

all_results_ensemble = []

for sku in df["sku_id"].unique():
    print(f"\n--- Training Ensemble for {sku} ---")
    sku_df = df[df["sku_id"] == sku].sort_values('date').reset_index(drop=True)
    
    # 2. CHRONOLOGICAL SPLIT (60% Train, 20% Meta-Train, 20% Test)
    # We need a holdout set to train the Meta-Model so it doesn't overfit
    train_idx = int(len(sku_df) * 0.6)
    meta_idx = int(len(sku_df) * 0.8)
    
    train_df = sku_df.iloc[:train_idx]
    meta_df = sku_df.iloc[train_idx:meta_idx]
    test_df = sku_df.iloc[meta_idx:]
    
    # 3. INITIALIZE LEVEL 0 BASE MODELS
    model_cat = CatBoostRegressor(iterations=150, learning_rate=0.1, depth=6, verbose=0, task_type="GPU")
    model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, tree_method="hist", device="cuda")
    model_lgb = LGBMRegressor(n_estimators=100, learning_rate=0.1, n_jobs=-1, verbose=-1)
    
    # Train Base Models on the 60% training set
    print("Training Level 0 Models (CatBoost, XGBoost, LightGBM)...")
    model_cat.fit(train_df[feature_cols_cat], train_df["units_sold_diff"], cat_features=cat_features_indices)
    model_xgb.fit(train_df[feature_cols_xgb_lgb], train_df["units_sold_diff"])
    model_lgb.fit(train_df[feature_cols_xgb_lgb], train_df["units_sold_diff"])
    
    # 4. GENERATE PREDICTIONS FOR THE META-MODEL
    # Base models predict on the 20% Meta Set
    meta_preds_cat = model_cat.predict(meta_df[feature_cols_cat])
    meta_preds_xgb = model_xgb.predict(meta_df[feature_cols_xgb_lgb])
    meta_preds_lgb = model_lgb.predict(meta_df[feature_cols_xgb_lgb])
    
    # Create the training dataset for the Level 1 Meta-Model
    X_meta_train = np.column_stack((meta_preds_cat, meta_preds_xgb, meta_preds_lgb))
    y_meta_train = meta_df["units_sold_diff"]
    
    # 5. TRAIN LEVEL 1 META-MODEL (Linear Regression)
    print("Training Level 1 Meta-Model (Linear Regression)...")
    meta_model = LinearRegression()
    meta_model.fit(X_meta_train, y_meta_train)
    
    # Print the weights it assigned to each model
    print(f"Algorithm Weights -> CatBoost: {meta_model.coef_[0]:.2f}, XGBoost: {meta_model.coef_[1]:.2f}, LightGBM: {meta_model.coef_[2]:.2f}")
    
    # 6. FINAL EVALUATION ON THE 20% TEST SET
    test_preds_cat = model_cat.predict(test_df[feature_cols_cat])
    test_preds_xgb = model_xgb.predict(test_df[feature_cols_xgb_lgb])
    test_preds_lgb = model_lgb.predict(test_df[feature_cols_xgb_lgb])
    
    X_test_meta = np.column_stack((test_preds_cat, test_preds_xgb, test_preds_lgb))
    
    # The Meta-Model makes the final unified prediction
    final_preds_diff = meta_model.predict(X_test_meta)
    
    # Reconstruct actual sales and floor at 0
    final_preds_actual = np.maximum(0, test_df["lag_1"].values + final_preds_diff)
    y_test_actual = test_df["units_sold"].values
    
    # Calculate Final Metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual, final_preds_actual))
    r2 = r2_score(y_test_actual, final_preds_actual)
    
    all_results_ensemble.append({"model": "Stacking_Ensemble_1M", "sku_id": sku, "RMSE": rmse, "R2": r2})

# 7. PRINT FINAL REPORT
print("\n=== FINAL ENSEMBLE PERFORMANCE ===")
summary_df = pd.DataFrame(all_results_ensemble).groupby("model")[["RMSE", "R2"]].mean().round(3)
print(summary_df)

Loading 1-Million-Row Dataset...

--- Training Ensemble for MBA13 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.70, XGBoost: 0.06, LightGBM: 0.24

--- Training Ensemble for MBA15 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.66, XGBoost: 0.16, LightGBM: 0.19

--- Training Ensemble for MBP14 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.94, XGBoost: 0.19, LightGBM: -0.13

--- Training Ensemble for MBP16 ---
Training Level 0 Models (CatBoost, XGBoost, LightGBM)...
Training Level 1 Meta-Model (Linear Regression)...
Algorithm Weights -> CatBoost: 0.98, XGBoost: 0.10, LightGBM: -0.08

=== FINAL ENSEMBLE PERFORMANCE ===
                       RMSE     R2
model                             
Stacking_En